# RAG with LangChain & Pinecone (Local Embeddings Edition)

Retrieval-Augmented Generation using:
- **LLM:** GPT-4o-mini (via GitHub Models - FREE)
- **Embeddings:** all-MiniLM-L6-v2 (Local - FREE)
- **Vector Store:** Pinecone

---

## 1. Setup & Dependencies

In [1]:
%pip install -qU langchain langchain-openai langchain-pinecone langchain-community langchain-text-splitters pinecone beautifulsoup4 langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [1]:
import getpass
import os

if not os.environ.get("GITHUB_TOKEN"):
    os.environ["GITHUB_TOKEN"] = getpass.getpass("Enter your Fine-grained GitHub Token (github_pat_...): ")

if not os.environ.get("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API key: ")

print("Tokens configured. OpenAI key NOT required.")

Tokens configured. OpenAI key NOT required.


## 2. Initialize Components

In [2]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

# LLM via GitHub Models
model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com"
)

# Local Embeddings (Running on your machine)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print(f"LLM: GitHub Models ({model.model_name})")
print(f"Embeddings: Local HuggingFace (all-MiniLM-L6-v2)")

/mnt/data/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM: GitHub Models (gpt-4o-mini)
Embeddings: Local HuggingFace (all-MiniLM-L6-v2)


## 3. Pinecone Vector Store Configuration

In [3]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

index_name = "arep-lab04-rag-local"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384, # all-MiniLM-L6-v2 dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index: {index_name}")
else:
    print(f"Index already exists: {index_name}")

index = pc.Index(index_name)
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

print(f"Vector store ready.")

Created index: arep-lab04-rag-local
Vector store ready.


## 4. Indexing Phase

### 4.1 Load Documents

In [4]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)

docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"Total characters: {len(docs[0].page_content)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 document(s)
Total characters: 43047


### 4.2 Split Documents

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

all_splits = text_splitter.split_documents(docs)

print(f"Split into {len(all_splits)} chunks")

Split into 63 chunks


### 4.3 Store in Pinecone

In [6]:
document_ids = vector_store.add_documents(documents=all_splits)

print(f"Indexed {len(document_ids)} documents in Pinecone")

Indexed 63 documents in Pinecone


## 5. Retrieval & Generation Phase

### 5.1 Similarity Search

In [7]:
results = vector_store.similarity_search("What is task decomposition?", k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Content: {doc.page_content[:200]}...")

--- Result 1 ---
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outl...
--- Result 2 ---
Content: Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a s...
--- Result 3 ---
Content: Finite context length: The restricted context capacity limits the inclusion of historical information, detailed instructions, API call context, and responses. The design of the system has to work with...


### 5.2 RAG Agent

In [ ]:
from langchain.tools import tool

@tool
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized

from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the tool to retrieve context from the blog post."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

tools = [retrieve_context]
agent = create_tool_calling_agent(model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("RAG agent ready.")

ImportError: cannot import name 'create_tool_calling_agent' from 'langchain.agents' (/mnt/data/.venv/lib/python3.11/site-packages/langchain/agents/__init__.py)

### 5.3 Query Demo

In [ ]:
query = "What is task decomposition?"
response = agent_executor.invoke({"input": query})
print(response["output"])

## 6. Cleanup (Optional)

In [ ]:
# pc.delete_index(index_name)
# print(f"Deleted index: {index_name}")